In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../models', exist_ok=True)

# Load cleaned data from Phase 1
df = pd.read_csv('../data/cicids2017_clean.csv', low_memory=False)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Recreate attack_category (since we saved the raw Label)
df['attack_category'] = df['Label'].apply(lambda x:
    'Benign' if x == 'BENIGN'
    else 'DoS/DDoS' if x in ['DoS Hulk', 'DoS GoldenEye', 'DoS slowloris', 'DoS Slowhttptest', 'DDoS', 'Heartbleed']
    else 'Brute Force' if x in ['FTP-Patator', 'SSH-Patator']
    else 'Web Attack' if 'Web Attack' in x
    else 'Reconnaissance' if x == 'PortScan'
    else 'Botnet' if x == 'Bot'
    else 'Infiltration' if x == 'Infiltration'
    else 'Unknown'
)

print(f"\nCategories:")
print(df['attack_category'].value_counts())

Loaded: 2520798 rows, 80 columns

Categories:
attack_category
Benign            2095057
DoS/DDoS           321770
Reconnaissance      90694
Brute Force          9150
Web Attack           2143
Botnet               1948
Infiltration           36
Name: count, dtype: int64


In [4]:
# SEPARATE FEATURES AND TARGET
#
# Drop 'Label' (original 15-class label) — we use attack_category
# Drop 'attack_category' — that's our target, not a feature
#
# Unlike NSL-KDD, ALL features are already numerical
# No encoding needed — this is a big advantage of CICIDS2017

X = df.drop(columns=['Label', 'attack_category'])
y = df['attack_category']

print(f"Features: {X.shape[1]} columns, all numerical")
print(f"Target: {y.nunique()} classes")
print(f"\nFeature dtypes:")
print(X.dtypes.value_counts())

Features: 78 columns, all numerical
Target: 7 classes

Feature dtypes:
int64      54
float64    24
Name: count, dtype: int64


In [5]:
# Encode target labels
target_encoder = LabelEncoder()
target_encoder.fit(sorted(y.unique()))  # sorted for consistency
y_encoded = target_encoder.transform(y)

print("Target encoding:")
for i, name in enumerate(target_encoder.classes_):
    count = (y_encoded == i).sum()
    print(f"  {i} = {name:20s} ({count:,} samples)")

Target encoding:
  0 = Benign               (2,095,057 samples)
  1 = Botnet               (1,948 samples)
  2 = Brute Force          (9,150 samples)
  3 = DoS/DDoS             (321,770 samples)
  4 = Infiltration         (36 samples)
  5 = Reconnaissance       (90,694 samples)
  6 = Web Attack           (2,143 samples)


In [6]:
# STRATIFIED TRAIN/TEST SPLIT
#
# stratify=y_encoded ensures every class appears proportionally
# in both train and test. This is why CICIDS2017 will perform
# better than NSL-KDD — no surprise unseen attack types.
#
# 80% train, 20% test

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # THIS is the key — proportional split
)

print(f"Train: {X_train.shape[0]:,} samples")
print(f"Test:  {X_test.shape[0]:,} samples")
print(f"\nTrain distribution:")
for i, name in enumerate(target_encoder.classes_):
    train_count = (y_train == i).sum()
    test_count = (y_test == i).sum()
    print(f"  {name:20s}: train={train_count:,}  test={test_count:,}")

Train: 2,016,638 samples
Test:  504,160 samples

Train distribution:
  Benign              : train=1,676,045  test=419,012
  Botnet              : train=1,559  test=389
  Brute Force         : train=7,320  test=1,830
  DoS/DDoS            : train=257,416  test=64,354
  Infiltration        : train=29  test=7
  Reconnaissance      : train=72,555  test=18,139
  Web Attack          : train=1,714  test=429


In [7]:
# SCALE FEATURES
# Same as NSL-KDD — fit on train, transform both

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling done.")
print(f"Train mean (first 5): {X_train_scaled.iloc[:, :5].mean().round(2).tolist()}")
print(f"Train std  (first 5): {X_train_scaled.iloc[:, :5].std().round(2).tolist()}")

Scaling done.
Train mean (first 5): [0.0, 0.0, 0.0, 0.0, -0.0]
Train std  (first 5): [1.0, 1.0, 1.0, 1.0, 1.0]


In [8]:
# Save for Phase 3
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)
pd.Series(y_train, name='target').to_csv('../data/y_train.csv', index=False)
pd.Series(y_test, name='target').to_csv('../data/y_test.csv', index=False)

joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(target_encoder, '../models/target_encoder.joblib')

print("Saved:")
print("  data/X_train_scaled.csv")
print("  data/X_test_scaled.csv")
print("  data/y_train.csv")
print("  data/y_test.csv")
print("  models/scaler.joblib")
print("  models/target_encoder.joblib")
print("\nPhase 2 complete!")

Saved:
  data/X_train_scaled.csv
  data/X_test_scaled.csv
  data/y_train.csv
  data/y_test.csv
  models/scaler.joblib
  models/target_encoder.joblib

Phase 2 complete!
